# AI Portfolio Assistant with RAG

This notebook builds an AI portfolio assistant using Retrieval-Augmented Generation (RAG).

Main components:
- Loading and parsing local repositories
- Chunking documents
- Generating summaries with LLMs
- Creating embeddings and vector search
- Retrieval + reranking
- Tool calling
- Gradio chat interface
- Evaluation pipeline for retrieval and answers

The assistant answers questions about AI/ML projects based on repository content and LinkedIn data.


## Imports and Environment Setup
This section imports libraries used for embeddings, vector databases, OpenAI API calls, evaluation, and the Gradio interface.

In [1]:
# imports
from dotenv import load_dotenv
from openai import OpenAI
from chromadb import PersistentClient
from pydantic import BaseModel, Field
from litellm import completion
import json
import re
import os
import requests
from pypdf import PdfReader
import gradio as gr
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from collections import Counter
import math

## Configuration
Model names, database names, and collection settings are defined here.

In [2]:
MODEL = "gpt-4.1-mini"
openai = OpenAI()
DB_NAME = "vector_db_2"
collection_name = "docs"
PROJECT_COLLECTION = "project_summaries"
embedding_model = "text-embedding-3-small"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

RETRIEVAL_K = 20
FINAL_K = 8

OpenAI API Key exists and begins sk-proj-


## Notebook Parsing Utilities
Functions for loading `.ipynb` files and converting notebook cells into text for RAG indexing.

In [3]:
def load_ipynb(file_path):
    texts = []

    with open(file_path, "r", encoding="utf-8") as f:
        nb = json.load(f)

    for cell in nb.get("cells", []):
        if cell.get("cell_type") == "markdown":
            texts.append("MARKDOWN:\n" + "".join(cell.get("source", [])))
        elif cell.get("cell_type") == "code":
            texts.append("CODE:\n" + "".join(cell.get("source", [])))

    return "\n".join(texts)

## Project Metadata Extraction
Utilities for detecting project names and organizing repository structure.

In [4]:
def extract_project_name(path: str) -> str:
    if not path:
        return "UNKNOWN_PROJECT"

    path = path.replace("\\", "/")

    match = re.search(r"repos/([^/]+)/", path)

    if match:
        return match.group(1)

    return "UNKNOWN_PROJECT"

## Repository Loading
Loads source files from repositories and converts them into LangChain documents.

In [5]:
def load_files():
    documents = []

    repo_path = "repos"

    for root, _, files in os.walk(repo_path):
        if ".git" in root:
            continue

        print("Checking:", root)

        for file in files:
            print("Found:", file)

            if file.endswith((".md", ".ipynb")):
                file_path = os.path.join(root, file)

                try:
                    if file.endswith(".ipynb"):
                        content = load_ipynb(file_path)
                    else:
                        with open(file_path, "r", encoding="utf-8") as f:
                            content = f.read()

                    project_name = extract_project_name(file_path)

                    documents.append(
                        Document(
                            page_content=content,
                            metadata={
                                "source": file_path,
                                "project": project_name
                            }
                        )
                    )

                except Exception as e:
                    print("ERROR:", file_path, e)

    print(f"Loaded {len(documents)} documents")
    return documents

In [6]:
documents = load_files()

Checking: repos
Checking: repos\Career-chat-rag
Checking: repos\Car_brand_classification_CNN
Found: Car_brand_classification_CNN.ipynb
Found: README.md
Checking: repos\Cracow_Real_Estate_Price_Prediction_2025
Found: distance.csv
Found: Flats_feature_engineering.ipynb
Found: flats_result_set1.csv
Found: Flats_scraping.ipynb
Found: flats_set1.csv
Found: Models_results.ipynb
Found: README.md
Found: requirements.txt
Checking: repos\LIDAR_point_cloud_classification_ANN
Found: 78989_1475655_M-34-64-D-d-2-3-2-4.rar
Found: Lidar_point_cloud_classification.ipynb
Found: README.md
Checking: repos\NLP-Classification_of_tweets_about_airlines_LSTM
Found: Airlines_Sentiments_Analysis_LSTM.ipynb
Found: README.md
Found: training_twitter_x_y_train.csv
Loaded 10 documents


In [7]:
total_chars = sum(len(doc.page_content) for doc in documents)
print("Total characters:", total_chars)

chars_list= []
for doc in documents:
    chars_list.append(len(doc.page_content))
print("Characters in files:",chars_list)


Total characters: 113798
Characters in files: [16646, 738, 43359, 7645, 18526, 1032, 11684, 1266, 12087, 815]


## Chunking Strategy
Documents are split into overlapping chunks before embedding generation.

In [8]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 155 chunks
First chunk:

page_content='CODE:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping 
from tensorflow.keras.layers import Dense, Conv2D, Flatten, MaxPool2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from sklearn.metrics import confusion_matrix
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')
CODE:
# Defining training, validation and test sets
train_data_dir = 'Data/Train/' 
valid_data_dir = 'Data/Validation/'
test_data_dir = 'Data/Test/'
CODE:
#Defining the image size, batch size and number of output classes.
h, w = 256, 256 
batch_size = 8 
n_classes = 8
MARKDOWN:
# Convolution neural netwo

## Chunk Summarization
Each chunk is summarized using an LLM to improve retrieval quality and context understanding.

In [9]:
class Result(BaseModel):
    page_content: str
    metadata: dict

class Chunk(BaseModel):
    headline: str
    summary: str
    original_text: str

    def as_result(self, document):
        return Result(
            page_content=f"{self.headline}\n\n{self.summary}\n\n{self.original_text}",
            metadata=document.metadata
        )

In [10]:
def make_prompt(chunk_text):
    return f"""
You are an expert AI assistant summarizing technical code and markdown.

Your task:
- Create a short headline
- Create a short summary

IMPORTANT RULES:
- Use ONLY the provided text
- Keep summary concise (2-3 sentences)
- Return ONLY valid JSON

{{
  "headline": "short title",
  "summary": "short summary"
}}

TEXT:
{chunk_text}
"""

In [11]:
# Create embeddings and persist them into ChromaDB

def process_chunk(chunk, retries=3):
    for i in range(retries):
        try:
            prompt = make_prompt(chunk.page_content)

            response = completion(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
                temperature=0
            )

            content = response.choices[0].message.content

            data = json.loads(content)

            parsed = Chunk(
                headline=data["headline"],
                summary=data["summary"],
                original_text=chunk.page_content
            )

            return parsed.as_result(chunk)

        except Exception as e:
            print(f"Retry {i+1}: {e}")

    return Result(
        page_content=f"""TITLE: FALLBACK

SUMMARY: Failed to parse structured chunk, using raw content

CONTENT:
{chunk.page_content[:1000]}""",
        metadata=chunk.metadata
    )

In [12]:
def process_all_chunks(chunks):
    return [process_chunk(c) for c in chunks]

In [13]:
chunks_result = [process_chunk(chunk) for chunk in chunks]

## Project-Level Summaries
Chunk summaries are grouped into project summaries used later in RAG prompts.

In [14]:
def group_by_project(results):
    grouped = {}

    for r in results:
        project = r.metadata.get("project", "UNKNOWN_PROJECT")

        if project not in grouped:
            grouped[project] = []

        grouped[project].append(r)

    return grouped

In [15]:
# Rerank retrieved chunks using an LLM-based ranking approach

def generate_project_summary(project_name, context):

    prompt = f"""
You are analyzing an AI project.

Project name:
{project_name}

Your task:
Create structured JSON summary.

IMPORTANT:
If machine learning or deep learning models appear in the project,
include ALL detected models in "models_used".

Return ONLY valid JSON:

{{
    "project_name": "",
    "short_description": "",
    "main_goal": "",
    "tech_stack": [],
    "models_used": []
}}

PROJECT CONTENT:
{context}
"""

    response = completion(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0
    )

    return json.loads(response.choices[0].message.content)

In [16]:
def build_project_summaries_cache(chunks):

    grouped = group_by_project(chunks)

    cache = {}

    for project_name, items in grouped.items():

        context = "\n".join(
            c.page_content for c in items
        )

        summary = generate_project_summary(project_name, context)

        cache[project_name] = summary

    return cache

In [17]:
project_summaries_cache = build_project_summaries_cache(chunks_result) 

## Embedding Generation
Embeddings are created and stored in ChromaDB.

In [18]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    emb = openai.embeddings.create(model=embedding_model, input=texts).data
    vectors = [e.embedding for e in emb]

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")

In [19]:
# Main RAG answering pipeline

create_embeddings(chunks_result)

Vectorstore created with 155 documents


In [20]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)

## Reranking
Retrieved chunks are reranked using an LLM-based ranking step.

In [21]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [22]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You must rank all chunks by relevance.

STRICT RULES:
- Use ONLY indices from 1 to N
- Do NOT invent indices
- Do NOT skip any
- Do NOT repeat any
- Return exactly N indices

Return JSON:
{"order": [1,2,3,...]}
"""

    user_prompt = f"Question:\n{question}\n\nChunks:\n"

    for i, chunk in enumerate(chunks, start=1):
        user_prompt += f"\n[{i}] {chunk.page_content[:300]}"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    response = completion(
        model=MODEL,
        messages=messages,
        response_format=RankOrder
    )

    reply = response.choices[0].message.content

    try:
        order = RankOrder.model_validate_json(reply).order
    except:
        return chunks  

    seen = set()
    ranked = []

    for i in order:
        if isinstance(i, int) and 1 <= i <= len(chunks) and i not in seen:
            ranked.append(chunks[i - 1])
            seen.add(i)

    for idx in range(len(chunks)):
        if (idx + 1) not in seen:
            ranked.append(chunks[idx])

    return ranked

## RAG Prompt Design
Defines the main RAG system prompt and helper formatting functions.

In [23]:
rag_system_prompt = """
You answer questions about GitHub projects.

RULES:
- Use ONLY provided context
- Never invent projects
- Group information by project
- If project is missing in context → it does not exist
- All projects in context belong to Joanna Waligóra (her GitHub portfolio)

Each chunk contains:
- project name
- source file

Project summaries contain:
- project description
- goal
- tech stack
- models used

Use summaries for:
- project overview
- listing projects
- technology questions

Use chunks for:
- implementation details
- architecture
- code behavior

 PROJECT SUMMARIES:
{project_summaries}

CONTEXT:
{context}
"""

In [24]:
def format_project_summaries(cache):
    return "\n\n".join(
        f"""
PROJECT: {p.get("project_name", "")}
DESCRIPTION: {p.get("short_description", "")}
GOAL: {p.get("main_goal", "")}
TECH STACK: {", ".join(p.get("tech_stack", []))}
MODELS: {", ".join(p.get("models_used", []))}
""".strip()
        for p in cache.values()
    )

In [25]:
def make_rag_messages(question, history, chunks):

    project_summaries = format_project_summaries(project_summaries_cache) 

    context = "\n\n".join(
        f"[PROJECT: {c.metadata.get('project','UNKNOWN')}]\n"
        f"{c.page_content}"
        for c in chunks
    )

    system_prompt = rag_system_prompt.format(
        project_summaries=project_summaries,
        context=context
    )

    return [
        {"role": "system", "content": system_prompt},
        *history,
        {"role": "user", "content": question}
    ]

In [26]:
def fetch_context_unranked(question):
    query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

In [27]:
def fetch_context(original_question):
    chunks = fetch_context_unranked(original_question)
    reranked = rerank(original_question, chunks)
    return reranked[:FINAL_K]

In [28]:
def answer_question(question: str, history: list[dict] = None):
    if history is None:
        history = []

    chunks = fetch_context(question)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)

    return response.choices[0].message.content, chunks

## Additional Tools
Utility tools for notifications, logging, and user information handling.

In [29]:
def push(text):
    requests.post(
        "https://api.pushover.net/1/messages.json",
        data={
            "token": os.getenv("PUSHOVER_TOKEN"),
            "user": os.getenv("PUSHOVER_USER"),
            "message": text,
        }
    )

In [30]:
def record_user_details(email:str, name:str="Name not provided", notes:str="not provided")-> dict:
    """Store user contact details and notify via push"""
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return {"recorded": "ok"}

In [31]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it"
            }
            ,
            "notes": {
                "type": "string",
                "description": "Any additional information about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [32]:
def record_unknown_question(question: str)-> dict:
    """Store unknown user question and notify via push"""
    push(f"Recording {question} asked that I couldn't answer")
    return {"recorded": "ok"}

In [33]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            },
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

## Tool-Based RAG Function
Main callable function used by the assistant for answering questions with RAG.

In [34]:
def answer_with_rag(question: str, history: list[dict] = None):
    if history is None:
        history = []

    answer, chunks = answer_question(question, history)

    return {
        "answer": answer,
        "context_snippets": [c.page_content[:300] for c in chunks],
        "sources": [c.metadata.get("source", "unknown") for c in chunks]
    }

In [35]:
answer_with_rag_json = {
    "name": "answer_with_rag",
    "description": "Answer user questions using RAG over Joanna Waligóra projects. Returns answer and retrieved context snippets and sources.",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The user's question"
            },
            "history": {
                "type": "array",
                "description": "Chat history in OpenAI format (role/content dicts)",
                "items": {
                    "type": "object",
                    "properties": {
                        "role": {"type": "string"},
                        "content": {"type": "string"}
                    },
                    "required": ["role", "content"],
                    "additionalProperties": False
                }
            }
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [36]:
tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json},
        {"type": "function", "function": answer_with_rag_json }]

In [37]:
def handle_tool_calls(tool_calls):
    results = []

    for tool_call in tool_calls:
        tool_name = tool_call.function.name

        try:
            arguments = json.loads(tool_call.function.arguments)
        except:
            arguments = {}

        print(f"Tool called: {tool_name}", flush=True)

        tool = globals().get(tool_name)

        if tool is None:
            result = {"error": f"Tool {tool_name} not found"}
        else:
            try:
                result = tool(**arguments)
            except Exception as e:
                result = {"error": str(e)}

        results.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(result)
        })

    return results

## LinkedIn Context Loading
LinkedIn PDF content is extracted and injected into the assistant system prompt.

In [ ]:
reader = PdfReader("linkedin.pdf")

linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

name = "Joanna Waligóra"

## Main Assistant Prompt
Defines the behavior and rules for the portfolio assistant.

In [39]:
main_system_prompt = f"""
You are Joanna Waligóra's portfolio assistant.

{linkedin}

RULES:
- LinkedIn = career truth
- RAG = GitHub project truth
- For project questions ALWAYS use RAG first
- Never invent projects or skills
- If information is missing → say "I don't know"

Use RAG for:
- projects
- tech stack
- models
- implementation details
- portfolio questions

Be concise and factual.
"""

## Chat Interface
Creates the Gradio interface for interacting with the assistant.

In [40]:
def chat(message, history):
    messages = [{"role": "system", "content": main_system_prompt}]

    for h in history:
        messages.append({
            "role": "assistant" if h["role"] == "assistant" else "user",
            "content": h["content"]
        })

    messages.append({
        "role": "user",
        "content": message
    })

    for _ in range(2):

        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )

        msg = response.choices[0].message

        if msg.tool_calls:

            messages.append(msg)

            tool_results = handle_tool_calls(msg.tool_calls)

            messages.extend(tool_results)

        else:
            return msg.content or "No response generated."

    return "Tool loop limit reached."

In [41]:
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


# Evaluation Pipeline
This section evaluates both retrieval quality and answer quality.

In [42]:
TEST_FILE = "tests.jsonl"

In [43]:
class TestQuestion(BaseModel):
    """A test question with expected keywords and reference answer."""

    question: str = Field(description="The question to ask the RAG system")
    reference_answer: str = Field(description="The reference answer for this question")
    category: str = Field(description="Question category (e.g., atomic_fact, cross_context, comparative_reasoning)")
    keywords: list[str] = Field(description="Keywords that must appear in retrieved context")


In [44]:
def load_tests() -> list[TestQuestion]:
    """Load test questions from JSONL file."""
    tests = []
    with open(TEST_FILE, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line.strip())
            tests.append(TestQuestion(**data))
    return tests

In [45]:
tests = load_tests()

In [46]:
example = tests[0]
print(example.question)
print(example.category)
print(example.reference_answer)
print(example.keywords)

In Car_brand_classification_CNN, what batch size and image dimensions are used for the training and validation data generators?
atomic_fact
The batch size used is 8 and the image dimensions are 256x256 pixels for both training and validation data generators.
['batch size', 'image dimensions', 'training', 'validation', 'data generators']


In [47]:
count = Counter([t.category for t in tests])
count

Counter({'atomic_fact': 101,
         'comparative_reasoning': 18,
         'local_reasoning': 11,
         'cross_context': 3})

## Retrieval Metrics
Implements metrics such as MRR, nDCG, and coverage.

In [48]:
def calculate_mrr(keyword, docs):
    keyword = keyword.lower()
    for i, doc in enumerate(docs, start=1):
        if keyword in doc.page_content.lower():
            return 1 / i
    return 0.0


def calculate_dcg(relevances, k):
    return sum(rel / math.log2(i + 2) for i, rel in enumerate(relevances[:k]))


def calculate_ndcg(keyword, docs, k=10):
    keyword = keyword.lower()
    relevances = [1 if keyword in doc.page_content.lower() else 0 for doc in docs[:k]]
    
    dcg = calculate_dcg(relevances, k)
    idcg = calculate_dcg(sorted(relevances, reverse=True), k)
    
    return dcg / idcg if idcg > 0 else 0.0

In [49]:
def evaluate_retrieval_one(test):
    docs = fetch_context(test.question)

    mrrs = [calculate_mrr(k, docs) for k in test.keywords]
    ndcgs = [calculate_ndcg(k, docs) for k in test.keywords]

    return {
        "mrr": sum(mrrs)/len(mrrs),
        "ndcg": sum(ndcgs)/len(ndcgs),
        "coverage": sum(1 for x in mrrs if x > 0) / len(mrrs) * 100
    }

## Answer Evaluation
LLM-based evaluation of generated answers.

In [50]:
def evaluate_answer_one(test):
    answer, docs = answer_question(test.question)

    messages = [
        {
            "role": "system",
            "content": """
You are a STRICT evaluation system for RAG answers.

You MUST evaluate based on the REFERENCE ANSWER only.

IMPORTANT RULES:
- Do NOT use external knowledge.
- Do NOT guess facts.
- Only compare GENERATED ANSWER vs REFERENCE ANSWER.

You evaluate 3 independent dimensions:

1. accuracy:
- Is the final answer factually correct?
- 5 = exact correct final answer
- 1 = wrong answer

2. completeness:
- Does the answer include ALL key information from reference?
- 5 = fully covers reference answer
- 1 = missing most key info

3. relevance:
- Does the answer directly address the question?
- 5 = fully on-topic, no hallucinations
- 1 = off-topic or irrelevant

Return ONLY valid JSON in this exact format:
{
  "accuracy": int,
  "completeness": int,
  "relevance": int
}
"""
        },
        {
            "role": "user",
            "content": f"""
QUESTION:
{test.question}

REFERENCE ANSWER:
{test.reference_answer}

GENERATED ANSWER:
{answer}
"""
        }
    ]

    resp = completion(
        model="gpt-4.1-mini",
        messages=messages,
        response_format={"type": "json_object"}
    )

    try:
        scores = json.loads(resp.choices[0].message.content)

        return {
            "accuracy": scores.get("accuracy", 1),
            "completeness": scores.get("completeness", 1),
            "relevance": scores.get("relevance", 1),
        }

    except Exception:
        return {
            "accuracy": 1,
            "completeness": 1,
            "relevance": 1,
        }

## Full Evaluation Run
Runs retrieval and answer evaluation over the full test dataset.

In [51]:
# RETRIEVAL
mrr_total = 0
ndcg_total = 0
cov_total = 0

# ANSWERS
acc_total = 0
comp_total = 0
rel_total = 0

for test in tests:
    r = evaluate_retrieval_one(test)
    mrr_total += r["mrr"]
    ndcg_total += r["ndcg"]
    cov_total += r["coverage"]

    a = evaluate_answer_one(test)
    acc_total += a["accuracy"]
    comp_total += a["completeness"]
    rel_total += a["relevance"]

n = len(tests)

print("=== RETRIEVAL ===")
print("MRR:", round(mrr_total/n, 4))
print("nDCG:", round(ndcg_total/n, 4))
print("Coverage:", round(cov_total/n, 2), "%")

print("\n=== ANSWERS ===")
print("Accuracy:", round(acc_total/n, 2))
print("Completeness:", round(comp_total/n, 2))
print("Relevance:", round(rel_total/n, 2))

=== RETRIEVAL ===
MRR: 0.7285
nDCG: 0.7264
Coverage: 80.52 %

=== ANSWERS ===
Accuracy: 4.68
Completeness: 4.46
Relevance: 4.95


# Experiments and Optimization

## Objective

Optimize the RAG system for answering questions about AI and machine learning portfolio projects.

Evaluation metrics:

Retrieval:
- MRR
- nDCG
- Coverage

Answer quality:
- Accuracy
- Completeness
- Relevance

---

## Baseline

Initial version (`career_chat_1`):

- vector retrieval
- fixed chunking
- direct answer generation

Results:

| Metric | Score |
|---|---:|
| MRR | 0.3609 |
| nDCG | 0.4142 |
| Coverage | 64.33% |
| Accuracy | 4.53 |
| Completeness | 4.36 |
| Relevance | 4.89 |

This version served as the reference point for later optimization.

---

## Key Experiments

## Context Enrichment

A major improvement introduced in `career_chat_2` was enriching repository content before retrieval.

Two techniques were applied:

- chunk-level summaries generated for individual code and documentation chunks,
- project-level summaries describing repository goals, technologies, and machine learning models.

Observation:

- retrieval became more semantically meaningful,
- answer completeness improved,
- answers became more consistent across projects,
- cross-project reasoning improved.

A comparison between the baseline and the enriched version without reranking showed a substantial improvement in retrieval quality.

This indicated that structured repository summaries were already improving semantic matching before any reranking techniques were applied, contributing to more complete and consistent answers.

Decision:

✅ Keep

---

### Reranking

LLM reranking was introduced to improve retrieval ordering.

Observation:

Reranking significantly improved retrieval quality and retrieval consistency.

Decision:

✅ Keep

---
### Query Rewriting

Query rewriting was evaluated to improve retrieval recall.

Pipeline:

Question
→ Rewrite
→ Retrieval
→ Merge
→ Rerank

Observation:

- slightly improved coverage,
- increased pipeline complexity,
- produced only minor answer quality differences.

Final decision:

❌ Not included in the final pipeline

Implementation remains in the notebook for experimentation.

---

### Chunking

Different chunk sizes and overlaps were evaluated.

Final selection:

```python
chunk_size = 1000
chunk_overlap = 150
```

Observation:

- small chunks reduced context,
- large chunks introduced noise,
- medium-sized chunks achieved the best balance.

---

### Retrieval Configuration

Final configuration:

```python
RETRIEVAL_K = 20
FINAL_K = 8
```

Observation:

Retrieving more chunks improved recall but increased noise.

Decision:

✅ Retrieve Top 20 → Keep Top 8

---

## Final Architecture

Question
↓
Retriever (Top 20)
↓
Reranker
↓
Top 8
↓
LLM
↓
Answer

---

## Final Results

| Metric | Baseline | Final |
|---|---:|---:|
| MRR | 0.3609 | 0.7285 |
| nDCG | 0.4142 | 0.7264 |
| Coverage | 64.33% | 80.52% |
| Accuracy | 4.53 | 4.68 |
| Completeness | 4.36 | 4.46 |
| Relevance | 4.89 | 4.95 |

Retrieval quality improved substantially while maintaining strong answer quality.

---

## Conclusions

Main findings:

- context enrichment through chunk-level and project-level summaries significantly improved retrieval quality,
- project summaries improved cross-project reasoning and answer consistency,
- LLM reranking provided the largest additional gain in retrieval performance,
- optimized chunking reduced retrieval noise while preserving useful context,
- retrieval depth tuning improved grounding and coverage,
- query rewriting introduced additional complexity with only marginal benefits.

A key lesson from the experiments was that retrieval quality depends not only on retrieval algorithms but also on how repository knowledge is represented.

Enriching repository content with structured summaries improved semantic understanding of projects and enabled more complete and consistent answers. Combined with LLM reranking, this produced the strongest overall performance.

In [52]:
# =========================================================
# EXPERIMENTAL: QUERY REWRITING
# =========================================================
# Tested during RAG optimization.
# Improved recall slightly but reduced overall answer quality
# and introduced noisier retrieval results.
# Final production pipeline does NOT use rewriting.
# =========================================================

# def rewrite_query(question, history=[]):
#     """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
#     message = f"""
# You are answering questions about AI portfolio projects.

# This is the history of your conversation so far with the user:
# {history}

# And this is the user's current question:
# {question}

# Respond only with a short, refined question that you will use to search the Knowledge Base.
# It should be a VERY short specific question most likely to surface content. Focus on the question details.
# IMPORTANT: Respond ONLY with the precise knowledgebase query, nothing else.
# """
#     response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
#     return response.choices[0].message.content

# def merge_chunks(chunks, reranked):
#     merged = chunks[:]
#     existing = [chunk.page_content for chunk in chunks]
#     for chunk in reranked:
#         if chunk.page_content not in existing:
#             merged.append(chunk)
#     return merged

# def fetch_context(original_question):
#     rewritten_question = rewrite_query(original_question)
#     chunks1 = fetch_context_unranked(original_question)
#     chunks2 = fetch_context_unranked(rewritten_question)
#     chunks = merge_chunks(chunks1, chunks2)
#     reranked = rerank(original_question, chunks)
#     return reranked[:FINAL_K]